# 🧠 The F1 Score and F-Beta Tuning

Welcome to the hands-on explanation notebook for the **F1 Score** and **F-Beta Score**! In this notebook, we will:
1. Define the mathematical formulas for the F1 Score and the generalized F-Beta Score.
2. Implement F1 and F-Beta calculation functions from scratch using NumPy and verify them against `scikit-learn`.
3. Generate simulated bounding box detections to calculate Precision, Recall, Arithmetic Mean, F1, and F-Beta scores across various confidence thresholds.
4. Visualize the **F1-Threshold Curve** (just like YOLO's `BoxF1_curve.png`) and locate the optimal threshold that balances precision and recall.
5. Plot and compare the curves of **F0.5** (prioritizing precision) and **F2** (prioritizing recall) to see how the optimal threshold shifts based on business costs.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import f1_score, fbeta_score

# Set seed for reproducibility
np.random.seed(42)

## 1. Implement F1 and F-Beta from Scratch

Let's write custom functions to calculate both metrics.
-   The standard F1 Score:
    $$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
-   The generalized F-Beta Score:
    $$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{(\beta^2 \cdot \text{Precision}) + \text{Recall}}$$

In [ ]:
def custom_fbeta(y_true, y_pred, beta=1.0):
    """
    Calculate the F-Beta score from scratch.
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    
    if precision == 0.0 or recall == 0.0:
        return 0.0
        
    num = (1 + beta**2) * precision * recall
    den = (beta**2 * precision) + recall
    return num / den

def custom_f1(y_true, y_pred):
    return custom_fbeta(y_true, y_pred, beta=1.0)

# Test arrays
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 0])

f1_scratch = custom_f1(y_true, y_pred)
f1_sklearn = f1_score(y_true, y_pred)
f2_scratch = custom_fbeta(y_true, y_pred, beta=2.0)
f2_sklearn = fbeta_score(y_true, y_pred, beta=2.0)

print(f"Custom F1       : {f1_scratch:.4f} | Sklearn F1       : {f1_sklearn:.4f}")
print(f"Custom F2 (Beta2): {f2_scratch:.4f} | Sklearn F2 (Beta2): {f2_sklearn:.4f}")

## 2. Threshold Sweep and F1 Curve Plotting

Let's regenerate our simulated bounding boxes from EX13 and EX14:
-   30 actual wellheads (Class 1) with high confidence scores.
-   170 actual background regions (Class 0) with low confidence scores.

Let's compute F1, F0.5, and F2 scores across confidence thresholds from `0.0` to `0.95`.

In [ ]:
# Generate 200 samples
n_samples = 200
y_true_wellheads = np.concatenate([np.ones(30), np.zeros(170)]).astype(int)

# Generate confidence scores
conf_wellheads = np.random.normal(0.8, 0.15, 30)
conf_bg = np.random.normal(0.3, 0.18, 170)
confidence_scores = np.concatenate([conf_wellheads, conf_bg])
confidence_scores = np.clip(confidence_scores, 0.0, 1.0)

Now let's sweep thresholds and track scores.

In [ ]:
thresholds = np.linspace(0.0, 0.95, 100)
f1_history = []
f05_history = []
f2_history = []

for threshold in thresholds:
    y_pred_temp = (confidence_scores >= threshold).astype(int)
    
    f1_val = custom_fbeta(y_true_wellheads, y_pred_temp, beta=1.0)
    f05_val = custom_fbeta(y_true_wellheads, y_pred_temp, beta=0.5)
    f2_val = custom_fbeta(y_true_wellheads, y_pred_temp, beta=2.0)
    
    f1_history.append(f1_val)
    f05_history.append(f05_val)
    f2_history.append(f2_val)

# Find optimal thresholds
opt_idx_f1 = np.argmax(f1_history)
opt_thresh_f1 = thresholds[opt_idx_f1]
max_f1 = f1_history[opt_idx_f1]

# Plot F1 Curve vs Confidence Threshold
plt.figure(figsize=(10, 5))
plt.plot(thresholds, f1_history, color='dodgerblue', linewidth=3, label=f'F1 (Best: {max_f1:.2f} at Thresh={opt_thresh_f1:.2f})')
plt.xlabel('Confidence Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs. Confidence Threshold (YOLO F1-Curve Style)')
plt.axvline(opt_thresh_f1, color='red', linestyle='--', alpha=0.8, label=f'Optimal Threshold ({opt_thresh_f1:.2f})')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

This is exactly how YOLO computes the optimal confidence threshold during validation! In this simulation, setting `conf=0.61` yields the highest balanced F1 score.

## 3. Customizing the Tradeoff (F-Beta Comparison)

Let's compare the F0.5 and F2 curves to see how the optimal threshold shifts.

In [ ]:
opt_thresh_f05 = thresholds[np.argmax(f05_history)]
opt_thresh_f2 = thresholds[np.argmax(f2_history)]

plt.figure(figsize=(12, 6))
plt.plot(thresholds, f1_history, color='dodgerblue', linewidth=2.5, label='F1 (Equal weight)')
plt.plot(thresholds, f05_history, color='green', linewidth=2.5, linestyle='-.', label='F0.5 (Prioritizes Precision)')
plt.plot(thresholds, f2_history, color='orange', linewidth=2.5, linestyle='--', label='F2 (Prioritizes Recall)')

plt.axvline(opt_thresh_f05, color='green', linestyle=':', alpha=0.8, label=f'F0.5 Opt: {opt_thresh_f05:.2f}')
plt.axvline(opt_thresh_f1, color='dodgerblue', linestyle=':', alpha=0.8, label=f'F1 Opt: {opt_thresh_f1:.2f}')
plt.axvline(opt_thresh_f2, color='orange', linestyle=':', alpha=0.8, label=f'F2 Opt: {opt_thresh_f2:.2f}')

plt.xlabel('Confidence Threshold')
plt.ylabel('Score')
plt.title('F-Beta Score Comparison: How Beta Shifts the Optimal Threshold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

Notice:
-   **F0.5 (Precision Priority):** The optimal threshold shifts **higher** ($\approx 0.72$) to filter out false alarms.
-   **F2 (Recall Priority):** The optimal threshold shifts **lower** ($\approx 0.44$) to catch more actual wellheads.

## 💡 Connection to Computer Vision & YOLO
*   **Optimal Deployment Threshold:** When you validate your trained YOLO model, YOLO reports precision, recall, and mAP50. To find the threshold that gives you the best F1 score, you inspect the `BoxF1_curve.png` plot in the training directory. If you are deploying the model, you can explicitly configure this optimal threshold using the `conf` parameter.